<a href="https://colab.research.google.com/github/rudraroy1555/resume-screener-ranking/blob/main/tf_idf_resume_screener.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Candidate Ranking Engine via NLP
**Objective:** An unsupervised model to rank candidate resumes against a specific job description.

In [98]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [99]:
from google.colab import drive
drive.mount('/content/drive')

#Target Job Description
job_desc = """
Looking for a software engineer with strong Python skills.
Must have experience with data analysis, pandas, and machine learning.
"""
#Loding data
try:
    df = pd.read_csv("/content/drive/MyDrive/candidates_input.csv")
    print("--- Job Description ---")
    print(job_desc.strip())
    print(f"\n--- Successfully loaded {len(df)} candidates from Drive ---")
    print(df.head())#top 10
except :
    print(f"ERROR: Could not find '{"/content/drive/MyDrive/candidates_input.csv"}'.")
    print("Please ensure the file is in your Drive and named correctly.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Job Description ---
Looking for a software engineer with strong Python skills.
Must have experience with data analysis, pandas, and machine learning.

--- Successfully loaded 4 candidates from Drive ---
  candidate_name                                           raw_text
0       Alice_DS  Skills * Programming Languages: Python (pandas...
1         Bob_DS  Education Details \nMay 2013 to May 2017 B.E  ...
2     Charlie_HR  TECHNICAL SKILLS â¢ Typewriting â¢ TORA â¢ ...
3      Diana_Web  Technical Skills Web Technologies: Angular JS,...


## 1. Text Preprocessing
Standardize raw text by stripping punctuation, converting to lowercase, removing stop words, and lemmatizing.

In [100]:
nltk.download('stopwords', quiet=True)#quite helps to remove output from terminal
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

lemmatize =WordNetLemmatizer()
stop_words =set(stopwords.words('english'))
print(f"Number of stop words loaded: {len(stop_words)}")

def preprocess(text):
  # Lowercase, remove non-alphabetic characters, and tokenize
  tokens = [word for word in nltk.word_tokenize(text.lower()) if word.isalpha()]
  # Lemmatize and remove stopwords
  return " ".join(lemmatize.lemmatize(word) for word in tokens if word not in stop_words)

Number of stop words loaded: 198


## 2. Filter Stage: LogisticRegression(supervised learning)
Before ranking, we feed resumes into a Logistic Regression model trained on historical data. This functions as a gatekeeper: it classifies the job category for each candidate and discards bad fits immediately. That saves computing power, so our scoring engine can concentrate solely on qualified candidates.”

In [101]:
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore') # Suppress sklearn warnings

print("Downloading training dataset...")

try:
    # HuggingFace Dataset
    df_train = pd.read_csv("https://huggingface.co/datasets/Unknown92/Resume_dataset/raw/main/UpdatedResumeDataSet.csv")
    df_train['cleaned_resume'] = df_train['Resume'].apply(preprocess)

    vectorizer = TfidfVectorizer()
    X_train = vectorizer.fit_transform(df_train['cleaned_resume'])
    y_train = df_train['Category']

    lr_model = LogisticRegression()
    lr_model.fit(X_train, y_train)
    print("Done")

except Exception as e:
    print(f"Failed to load or train on dataset: {e}")

Done


In [102]:
# target category must match a category from the HuggingFace dataset
target_category = "Data Science"

print(f"--- FILTERING PHASE: Target Category = {target_category} ---\n")

df['clean_text'] = df['raw_text'].apply(preprocess)
X_candidates = vectorizer.transform(df['clean_text'])

# Predicting catagories
predicted_categories = lr_model.predict(X_candidates)
df['predicted_category'] = predicted_categories

print("Model Predictions:")
for index, row in df.iterrows():
    print(f"{row['candidate_name']}: {row['predicted_category']}")

# Filter: Keep only candidates matching the target category
df_filtered = df[df['predicted_category'] == target_category].copy()

print(f"\nCandidates passed: {len(df_filtered)}")

--- FILTERING PHASE: Target Category = Data Science ---

Model Predictions:
Alice_DS: Data Science
Bob_DS: Data Science
Charlie_HR: HR
Diana_Web: Web Designing

Candidates passed: 2


## 3. Preprocessing for Ranking

In [103]:
clean_jd = preprocess(job_desc)
df_filtered['clean_text'] = df_filtered['raw_text'].apply(preprocess)
print("--- Cleaned Text ---")
print(df_filtered[['candidate_name', 'clean_text']] )

--- Cleaned Text ---
  candidate_name                                         clean_text
0       Alice_DS  skill programming language python panda numpy ...
1         Bob_DS  education detail may may data scientist data s...


In [104]:
ranking_vectorizer = TfidfVectorizer()
all_text = [clean_jd] + df_filtered['clean_text'].tolist()
tfidf_matrix = ranking_vectorizer.fit_transform(all_text)

feature_names = ranking_vectorizer.get_feature_names_out()
dense_matrix = tfidf_matrix.toarray()

matrix_df = pd.DataFrame(dense_matrix, columns=feature_names)

labels = ['Job Description'] + df_filtered['candidate_name'].tolist()
matrix_df.index = labels

print("-----------------TF-IDF Matrix----------------")
print(matrix_df.round(3).T)

-----------------TF-IDF Matrix----------------
               Job Description  Alice_DS  Bob_DS
accelerating               0.0     0.030   0.000
accounting                 0.0     0.030   0.000
across                     0.0     0.091   0.000
action                     0.0     0.030   0.000
address                    0.0     0.030   0.000
...                        ...       ...     ...
visualization              0.0     0.061   0.000
word                       0.0     0.152   0.000
worked                     0.0     0.023   0.054
year                       0.0     0.023   0.323
young                      0.0     0.061   0.000

[345 rows x 3 columns]


## 4. Cosine Similarity, Ranking, & Export
Calculate the geometric angle between the Job Description and Candidate vectors to score alignment, then export to CSV.

In [105]:
from google.colab import drive

# Scoring
vector = tfidf_matrix[0]
candidate_vectors = tfidf_matrix[1:]
scores = cosine_similarity(vector, candidate_vectors).flatten()

df_filtered['similarity_score'] = scores
df_ranked = df_filtered.sort_values(by='similarity_score', ascending=False)

print("--- Final Candidate Ranking ---")
print(df_ranked[['candidate_name', 'similarity_score', 'raw_text']])

# Exporting
export_df = df_ranked[['candidate_name', 'similarity_score', 'raw_text']]
file_path = "/content/drive/MyDrive/candidate_ranking_results.csv"
export_df.to_csv(file_path, index=False)
print(f"\nResults successfully exported to {file_path}")

--- Final Candidate Ranking ---
  candidate_name  similarity_score  \
0       Alice_DS          0.133963   
1         Bob_DS          0.049032   

                                            raw_text  
0  Skills * Programming Languages: Python (pandas...  
1  Education Details \nMay 2013 to May 2017 B.E  ...  

Results successfully exported to /content/drive/MyDrive/candidate_ranking_results.csv
